# Race Distance Parsing

## Bounded question

> How is race distance represented in the source, and which values can be parsed reproducibly?

## Source and grain

This notebook examines the immutable SQLite source database at:

`data/raw/form_2015-present/form_2015-present/raceform.db`

The source table is `data`. The imported header row at SQLite `rowid = 1` is excluded throughout using `rowid <> 1`. Evidence will be profiled primarily at provisional race grain, using `date + course + off`, while retaining the original runner-level rows and SQLite `rowid` for source lineage.

## Method

The study begins with the source as found. It will identify which column or columns encode scheduled race distance, inspect SQLite storage classes, missing values, sentinels and distinct notation patterns, and test whether values are consistent within provisional races. Parsing rules will be proposed only after the observed miles, furlongs, yards, metric, fractional and jurisdiction-specific conventions have been established. Exact raw values will remain separate from any detected notation, parsed components or candidate analytical distance.

## Explicit exclusions

This notebook concerns scheduled race-distance representation. It will not reopen `btn` and `ovr_btn` as a second beaten-distance study; their established behaviour from Notebook 05 is carried forward only as a dependency. It will not overwrite ambiguous source values, assume that every distance can be standardised, or design the final staging or target database schema.


In [1]:
# Import the standard libraries needed for source inspection.
from pathlib import Path
import sqlite3

import pandas as pd

# Resolve the project and immutable source-database paths.
PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
PROVISIONAL_RACE_COLUMNS = ["date", "course", "off"]

# Confirm that the expected source database is available.
assert SOURCE_DB_PATH.exists(), f"Source database not found: {SOURCE_DB_PATH}"

# Open SQLite explicitly in read-only mode.
connection = sqlite3.connect(f"file:{SOURCE_DB_PATH}?mode=ro", uri=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source database: {SOURCE_DB_PATH}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Data-row predicate: {DATA_ROW_PREDICATE}")
print("SQLite connection opened in read-only mode.")

Project root: /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
Source table: data
Data-row predicate: rowid <> 1
SQLite connection opened in read-only mode.


In [2]:
# Inspect the source-table schema before assuming which field represents
# scheduled race distance.
#
# PRAGMA table_info is metadata-only and does not modify the database.
schema_df = pd.read_sql_query(
    f"PRAGMA table_info({SOURCE_TABLE})",
    connection,
)

# Display all source columns so that the distance field is identified from
# evidence rather than from memory or naming assumptions.
schema_df

,cid,name,type,notnull,dflt_value,pk
0,0,date,NUMERIC,0,None,0
1,1,course,TEXT,0,None,0
2,2,race_id,INTEGER,0,None,0
3,3,off,TEXT,0,None,0
4,4,race_name,TEXT,0,None,0
5,5,type,TEXT,0,None,0
6,6,class,TEXT,0,None,0
7,7,pattern,TEXT,0,None,0
8,8,rating_band,TEXT,0,None,0
9,9,age_band,TEXT,0,None,0


In [3]:
# Profile the raw `dist` field at both runner-record and provisional-race grain.
#
# This establishes:
# - actual SQLite storage classes;
# - SQL NULLs and blank-string values;
# - the number of distinct raw representations;
# - whether each provisional race contains one consistent raw distance value.

distance_profile_query = f"""
WITH source_rows AS (
    SELECT
        date,
        course,
        off,
        dist,
        typeof(dist) AS dist_storage_class
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
race_distance_summary AS (
    SELECT
        date,
        course,
        off,
        COUNT(*) AS runner_records,
        COUNT(DISTINCT dist) AS distinct_non_null_distance_values,
        COUNT(DISTINCT COALESCE(CAST(dist AS TEXT), '<SQL_NULL>'))
            AS distinct_distance_values_including_null
    FROM source_rows
    GROUP BY date, course, off
)
SELECT
    (SELECT COUNT(*) FROM source_rows) AS runner_records,
    (SELECT COUNT(*) FROM race_distance_summary) AS provisional_races,
    (SELECT COUNT(*) FROM source_rows WHERE dist IS NULL) AS sql_null_rows,
    (
        SELECT COUNT(*)
        FROM source_rows
        WHERE dist IS NOT NULL
          AND TRIM(CAST(dist AS TEXT)) = ''
    ) AS blank_string_rows,
    (
        SELECT COUNT(DISTINCT CAST(dist AS TEXT))
        FROM source_rows
        WHERE dist IS NOT NULL
    ) AS distinct_raw_distance_values,
    (
        SELECT COUNT(*)
        FROM race_distance_summary
        WHERE distinct_distance_values_including_null = 1
    ) AS races_with_one_raw_distance_value,
    (
        SELECT COUNT(*)
        FROM race_distance_summary
        WHERE distinct_distance_values_including_null > 1
    ) AS races_with_inconsistent_raw_distance_values
"""

distance_profile_df = pd.read_sql_query(
    distance_profile_query,
    connection,
)

storage_class_df = pd.read_sql_query(
    f"""
    SELECT
        typeof(dist) AS dist_storage_class,
        COUNT(*) AS runner_records
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY typeof(dist)
    ORDER BY runner_records DESC
    """,
    connection,
)

display(distance_profile_df)
display(storage_class_df)

,runner_records,provisional_races,sql_null_rows,blank_string_rows,distinct_raw_distance_values,races_with_one_raw_distance_value,races_with_inconsistent_raw_distance_values
0,1851285,189043,0,0,63,189043,0


,dist_storage_class,runner_records
0,text,1851285


In [4]:
# List every distinct raw distance value at provisional-race grain.
#
# Counting races rather than runner rows prevents large-field races from
# disproportionately influencing the apparent frequency of a notation.

raw_distance_values_df = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            MIN(dist) AS raw_dist
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY date, course, off
    )
    SELECT
        raw_dist,
        COUNT(*) AS provisional_races,
        MIN(date) AS first_observed_date,
        MAX(date) AS last_observed_date
    FROM provisional_races
    GROUP BY raw_dist
    ORDER BY
        provisional_races DESC,
        raw_dist
    """,
    connection,
)

pd.set_option("display.max_rows", None)
raw_distance_values_df

,raw_dist,provisional_races,first_observed_date,last_observed_date
0,1m,21729,2015-01-01,2026-05-27
1,6f,20887,2015-01-01,2026-05-27
2,7f,19920,2015-01-02,2026-05-27
3,2m,13471,2015-01-01,2026-05-27
4,5f,11883,2015-01-01,2026-05-27
5,1m2f,11344,2015-01-01,2026-05-27
6,1m4f,8423,2015-01-01,2026-05-27
7,2m4f,6833,2015-01-01,2026-05-27
8,2m½f,5161,2015-01-02,2026-05-25
9,3m,4404,2015-01-01,2026-05-27


In [5]:
# Classify all 63 raw distance values against an explicit candidate grammar.
#
# Candidate grammar:
# - optional whole miles followed by "m";
# - optional whole furlongs, possibly followed by the Unicode half symbol "½";
# - a final "f" when a furlong component is present;
# - at least one of miles or furlongs must be present.
#
# This cell does not yet convert values into a standard unit. It tests whether
# one deterministic notation rule covers the complete observed source domain.

import re

DISTANCE_PATTERN = re.compile(
    r"""
    ^
    (?:
        (?P<miles>\d+)m
        (?:
            (?P<furlongs_after_miles>\d+)
            (?P<half_after_miles>½)?
            f
        )?
      |
        (?P<furlongs_only>\d+)
        (?P<half_furlongs_only>½)?
        f
    )
    $
    """,
    re.VERBOSE,
)


def classify_raw_distance(raw_dist: str) -> dict:
    """Test one exact source value against the candidate distance grammar."""
    match = DISTANCE_PATTERN.fullmatch(raw_dist)

    if match is None:
        return {
            "raw_dist": raw_dist,
            "matches_candidate_grammar": False,
            "miles_text": None,
            "furlongs_text": None,
            "has_half_furlong": None,
        }

    groups = match.groupdict()

    furlongs_text = (
        groups["furlongs_after_miles"]
        if groups["furlongs_after_miles"] is not None
        else groups["furlongs_only"]
    )

    has_half_furlong = bool(
        groups["half_after_miles"] or groups["half_furlongs_only"]
    )

    return {
        "raw_dist": raw_dist,
        "matches_candidate_grammar": True,
        "miles_text": groups["miles"],
        "furlongs_text": furlongs_text,
        "has_half_furlong": has_half_furlong,
    }


distance_grammar_df = pd.DataFrame(
    classify_raw_distance(raw_dist)
    for raw_dist in raw_distance_values_df["raw_dist"]
).merge(
    raw_distance_values_df[["raw_dist", "provisional_races"]],
    on="raw_dist",
    how="left",
    validate="one_to_one",
)

grammar_summary_df = pd.DataFrame(
    {
        "distinct_raw_values": [len(distance_grammar_df)],
        "matching_distinct_values": [
            int(distance_grammar_df["matches_candidate_grammar"].sum())
        ],
        "non_matching_distinct_values": [
            int((~distance_grammar_df["matches_candidate_grammar"]).sum())
        ],
        "matching_provisional_races": [
            int(
                distance_grammar_df.loc[
                    distance_grammar_df["matches_candidate_grammar"],
                    "provisional_races",
                ].sum()
            )
        ],
        "non_matching_provisional_races": [
            int(
                distance_grammar_df.loc[
                    ~distance_grammar_df["matches_candidate_grammar"],
                    "provisional_races",
                ].sum()
            )
        ],
    }
)

display(grammar_summary_df)
display(
    distance_grammar_df.loc[
        ~distance_grammar_df["matches_candidate_grammar"]
    ].sort_values("raw_dist")
)

,distinct_raw_values,matching_distinct_values,non_matching_distinct_values,matching_provisional_races,non_matching_provisional_races
0,63,59,4,178541,10502


,raw_dist,matches_candidate_grammar,miles_text,furlongs_text,has_half_furlong,provisional_races
10,1m½f,False,NaN,NaN,None,4141
8,2m½f,False,NaN,NaN,None,5161
33,3m½f,False,NaN,NaN,None,1176
55,4m½f,False,NaN,NaN,None,24


## Candidate grammar result

### Observation

The first candidate grammar matches 59 of the 63 distinct raw values, covering 178,541 of 189,043 provisional races.

The four non-matching values are:

- `1m½f`
- `2m½f`
- `3m½f`
- `4m½f`

Together they occur in 10,502 provisional races.

These values are regular source notation rather than malformed exceptions. They express a whole-mile component followed by one half-furlong, with the zero whole-furlong component omitted. In other words, `2m½f` represents two miles and half a furlong, not an ambiguous or failed value.

### Interpretation

The initial grammar was too restrictive because it required at least one digit before `½` whenever a furlong component followed miles.

The candidate source grammar should therefore allow all of the following:

- whole miles only, such as `2m`;
- whole furlongs only, such as `6f`;
- fractional furlongs only, such as `7½f`;
- miles plus whole furlongs, such as `2m3f`;
- miles plus whole and half furlongs, such as `2m3½f`;
- miles plus a half-furlong with the zero omitted, such as `2m½f`.

This is evidence that the candidate grammar needs revision, not evidence of four unresolved source values. The revised grammar must be tested against all 63 distinct values and all 189,043 provisional races before any conversion rule is accepted.

In [6]:
# Revise the candidate grammar so that a half-furlong may follow miles
# without an explicit zero whole-furlong component, as in "2m½f".
#
# The parser still preserves the exact raw value and only extracts components.
# Standard-unit conversion will be tested separately after full grammar coverage.

REVISED_DISTANCE_PATTERN = re.compile(
    r"""
    ^
    (?:
        (?P<miles>\d+)m
        (?:
            (?:
                (?P<furlongs_after_miles>\d+)
                (?P<half_after_miles>½)?
            )
            |
            (?P<half_only_after_miles>½)
        )?
        f?
      |
        (?P<furlongs_only>\d+)
        (?P<half_furlongs_only>½)?
        f
    )
    $
    """,
    re.VERBOSE,
)


def classify_revised_raw_distance(raw_dist: str) -> dict:
    """Extract source distance components using the revised candidate grammar."""
    match = REVISED_DISTANCE_PATTERN.fullmatch(raw_dist)

    if match is None:
        return {
            "raw_dist": raw_dist,
            "matches_revised_grammar": False,
            "miles": None,
            "whole_furlongs": None,
            "has_half_furlong": None,
        }

    groups = match.groupdict()

    miles = int(groups["miles"]) if groups["miles"] is not None else 0

    if groups["furlongs_after_miles"] is not None:
        whole_furlongs = int(groups["furlongs_after_miles"])
    elif groups["furlongs_only"] is not None:
        whole_furlongs = int(groups["furlongs_only"])
    else:
        whole_furlongs = 0

    has_half_furlong = bool(
        groups["half_after_miles"]
        or groups["half_only_after_miles"]
        or groups["half_furlongs_only"]
    )

    return {
        "raw_dist": raw_dist,
        "matches_revised_grammar": True,
        "miles": miles,
        "whole_furlongs": whole_furlongs,
        "has_half_furlong": has_half_furlong,
    }


revised_distance_grammar_df = pd.DataFrame(
    classify_revised_raw_distance(raw_dist)
    for raw_dist in raw_distance_values_df["raw_dist"]
).merge(
    raw_distance_values_df[["raw_dist", "provisional_races"]],
    on="raw_dist",
    how="left",
    validate="one_to_one",
)

revised_grammar_summary_df = pd.DataFrame(
    {
        "distinct_raw_values": [len(revised_distance_grammar_df)],
        "matching_distinct_values": [
            int(revised_distance_grammar_df["matches_revised_grammar"].sum())
        ],
        "non_matching_distinct_values": [
            int((~revised_distance_grammar_df["matches_revised_grammar"]).sum())
        ],
        "matching_provisional_races": [
            int(
                revised_distance_grammar_df.loc[
                    revised_distance_grammar_df["matches_revised_grammar"],
                    "provisional_races",
                ].sum()
            )
        ],
        "non_matching_provisional_races": [
            int(
                revised_distance_grammar_df.loc[
                    ~revised_distance_grammar_df["matches_revised_grammar"],
                    "provisional_races",
                ].sum()
            )
        ],
    }
)

display(revised_grammar_summary_df)
display(
    revised_distance_grammar_df.loc[
        ~revised_distance_grammar_df["matches_revised_grammar"]
    ].sort_values("raw_dist")
)

,distinct_raw_values,matching_distinct_values,non_matching_distinct_values,matching_provisional_races,non_matching_provisional_races
0,63,63,0,189043,0


,raw_dist,matches_revised_grammar,miles,whole_furlongs,has_half_furlong,provisional_races


## Revised grammar coverage

### Observation

The revised grammar matches all 63 distinct raw distance values and all 189,043 provisional races.

No raw distance value remains unmatched.

The observed source domain therefore supports one consistent notation family:

- whole miles;
- whole furlongs;
- miles plus furlongs;
- optional half-furlongs represented by `½`;
- omitted zero whole-furlongs in values such as `2m½f`.

There is no observed metric, yards-only, decimal, blank, sentinel or malformed distance notation in the current source extract.

### Interpretation

A deterministic component parser is now supported for the complete observed population.

This does not yet establish that every parsed component is semantically valid. The next test should check the extracted component ranges and determine whether any value uses eight or more furlongs alongside a mile component, which would indicate a non-canonical or potentially ambiguous representation.

### Candidate database consequence

The exact raw `dist` value should be preserved unchanged.

Separate candidate fields may be justified for:

- notation family;
- whole miles;
- whole furlongs;
- half-furlong indicator;
- parse status.

A standard analytical distance should not yet be accepted until component ranges and conversion reconciliation have been tested across the complete provisional-race population.

In [7]:
# Test whether the parsed components stay within canonical racing-distance ranges.
#
# In a canonical miles-plus-furlongs expression:
# - miles must be non-negative;
# - whole furlongs should be between 0 and 7;
# - the optional half-furlong adds 0.5 furlong;
# - at least one positive distance component must be present.
#
# This cell checks the complete distinct-value domain and weights any failures
# by the number of provisional races affected.

component_validation_df = revised_distance_grammar_df.copy()

component_validation_df["furlongs_component"] = (
    component_validation_df["whole_furlongs"]
    + component_validation_df["has_half_furlong"].astype(int) * 0.5
)

component_validation_df["total_furlongs"] = (
    component_validation_df["miles"] * 8
    + component_validation_df["furlongs_component"]
)

component_validation_df["miles_non_negative"] = (
    component_validation_df["miles"] >= 0
)

component_validation_df["whole_furlongs_in_canonical_range"] = (
    component_validation_df["whole_furlongs"].between(0, 7)
)

component_validation_df["positive_total_distance"] = (
    component_validation_df["total_furlongs"] > 0
)

component_validation_df["components_valid"] = (
    component_validation_df["matches_revised_grammar"]
    & component_validation_df["miles_non_negative"]
    & component_validation_df["whole_furlongs_in_canonical_range"]
    & component_validation_df["positive_total_distance"]
)

component_validation_summary_df = pd.DataFrame(
    {
        "distinct_raw_values": [len(component_validation_df)],
        "valid_distinct_values": [
            int(component_validation_df["components_valid"].sum())
        ],
        "invalid_distinct_values": [
            int((~component_validation_df["components_valid"]).sum())
        ],
        "valid_provisional_races": [
            int(
                component_validation_df.loc[
                    component_validation_df["components_valid"],
                    "provisional_races",
                ].sum()
            )
        ],
        "invalid_provisional_races": [
            int(
                component_validation_df.loc[
                    ~component_validation_df["components_valid"],
                    "provisional_races",
                ].sum()
            )
        ],
        "minimum_total_furlongs": [
            float(component_validation_df["total_furlongs"].min())
        ],
        "maximum_total_furlongs": [
            float(component_validation_df["total_furlongs"].max())
        ],
    }
)

display(component_validation_summary_df)

display(
    component_validation_df.loc[
        ~component_validation_df["components_valid"],
        [
            "raw_dist",
            "miles",
            "whole_furlongs",
            "has_half_furlong",
            "total_furlongs",
            "provisional_races",
        ],
    ].sort_values("raw_dist")
)

,distinct_raw_values,valid_distinct_values,invalid_distinct_values,valid_provisional_races,invalid_provisional_races,minimum_total_furlongs,maximum_total_furlongs
0,63,63,0,189043,0,4.0,36.5


,raw_dist,miles,whole_furlongs,has_half_furlong,total_furlongs,provisional_races


## Component validation

### Observation

All 63 distinct raw distance values, covering all 189,043 provisional races, produce valid canonical components.

The observed component rules are:

- whole miles are non-negative;
- whole furlongs remain within the canonical range `0` to `7`;
- half-furlongs are represented only by the `½` indicator;
- every parsed value produces a positive total distance.

The shortest observed scheduled distance is `4f`, equivalent to `4.0` furlongs.

The longest observed scheduled distance is `4m4½f`, equivalent to `36.5` furlongs.

No value requires component normalisation such as carrying eight furlongs into an additional mile.

### Interpretation

The complete observed source domain supports deterministic conversion into total furlongs without ambiguity or information loss, provided the exact raw `dist` value is retained separately.

The next step is to derive candidate standard distances in furlongs, yards and metres and verify exact arithmetic relationships across all distinct values and provisional races.

The metric value should be treated as a calculated analytical conversion from the source’s imperial notation, not as a metric distance supplied by the source.

In [8]:
# Derive candidate analytical distances from the validated components.
#
# Exact racing-distance relationships:
# - 1 mile = 8 furlongs
# - 1 furlong = 220 yards
# - 1 yard = 0.9144 metres
#
# Yards remain exact integers for all observed half-furlong values because
# half a furlong equals 110 yards. Metres are a calculated conversion.

candidate_distance_df = component_validation_df.copy()

candidate_distance_df["total_yards"] = (
    candidate_distance_df["total_furlongs"] * 220
).astype(int)

candidate_distance_df["total_metres"] = (
    candidate_distance_df["total_yards"] * 0.9144
)

# Reconstruct total furlongs independently from yards to verify that the
# conversion is reversible across the complete observed domain.
candidate_distance_df["reconstructed_total_furlongs"] = (
    candidate_distance_df["total_yards"] / 220
)

candidate_distance_df["conversion_reconciles"] = (
    candidate_distance_df["reconstructed_total_furlongs"]
    == candidate_distance_df["total_furlongs"]
)

conversion_summary_df = pd.DataFrame(
    {
        "distinct_raw_values": [len(candidate_distance_df)],
        "reconciling_distinct_values": [
            int(candidate_distance_df["conversion_reconciles"].sum())
        ],
        "non_reconciling_distinct_values": [
            int((~candidate_distance_df["conversion_reconciles"]).sum())
        ],
        "reconciling_provisional_races": [
            int(
                candidate_distance_df.loc[
                    candidate_distance_df["conversion_reconciles"],
                    "provisional_races",
                ].sum()
            )
        ],
        "non_reconciling_provisional_races": [
            int(
                candidate_distance_df.loc[
                    ~candidate_distance_df["conversion_reconciles"],
                    "provisional_races",
                ].sum()
            )
        ],
        "minimum_total_yards": [
            int(candidate_distance_df["total_yards"].min())
        ],
        "maximum_total_yards": [
            int(candidate_distance_df["total_yards"].max())
        ],
        "minimum_total_metres": [
            float(candidate_distance_df["total_metres"].min())
        ],
        "maximum_total_metres": [
            float(candidate_distance_df["total_metres"].max())
        ],
    }
)

display(conversion_summary_df)

display(
    candidate_distance_df.loc[
        ~candidate_distance_df["conversion_reconciles"],
        [
            "raw_dist",
            "total_furlongs",
            "total_yards",
            "reconstructed_total_furlongs",
            "provisional_races",
        ],
    ].sort_values("raw_dist")
)

,distinct_raw_values,reconciling_distinct_values,non_reconciling_distinct_values,reconciling_provisional_races,non_reconciling_provisional_races,minimum_total_yards,maximum_total_yards,minimum_total_metres,maximum_total_metres
0,63,63,0,189043,0,880,8030,804.672,7342.632


,raw_dist,total_furlongs,total_yards,reconstructed_total_furlongs,provisional_races


## Candidate analytical conversion

### Observation

All 63 distinct raw distance values, covering all 189,043 provisional races, convert reproducibly to total furlongs and total yards.

The conversion reconciles exactly in every case:

- `1 mile = 8 furlongs`;
- `1 furlong = 220 yards`;
- `½ furlong = 110 yards`.

The observed range is:

- minimum: `4f` = `880` yards = `804.672` metres;
- maximum: `4m4½f` = `8,030` yards = `7,342.632` metres.

No observed value produces a fractional yard, and converting yards back to furlongs reproduces the parsed total furlongs for the complete source domain.

### Interpretation

Total yards is the strongest candidate standard analytical distance because it is an exact integer representation for every observed source value.

Total furlongs is also reproducible, but half-furlong values require fractional numeric storage. Metres are deterministically calculable, but they are derived analytical conversions rather than values expressed by the source.

### Candidate database consequence

A later reversible transformation can preserve:

- exact raw `dist`;
- detected notation family;
- parsed miles;
- parsed whole furlongs;
- half-furlong indicator;
- candidate total furlongs;
- candidate total yards;
- optionally derived metres;
- parse status and validation flags.

Before accepting this rule, the parser should now be applied directly to every provisional race rather than only to the 63-value domain, with reconciliation checks for row coverage, uniqueness and exact raw-value preservation.

In [9]:
# Apply the validated parser to every provisional race and reconcile coverage.
#
# This moves the test from the 63-value domain to the complete race population.
# The exact raw value is retained alongside parsed components and candidate
# analytical distances.

provisional_race_distance_df = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        MIN(dist) AS raw_dist,
        COUNT(*) AS runner_records
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY date, course, off
    """,
    connection,
)

parsed_race_distance_df = pd.DataFrame(
    classify_revised_raw_distance(raw_dist)
    for raw_dist in provisional_race_distance_df["raw_dist"]
)

race_distance_result_df = pd.concat(
    [
        provisional_race_distance_df.reset_index(drop=True),
        parsed_race_distance_df.drop(columns=["raw_dist"]).reset_index(drop=True),
    ],
    axis=1,
)

race_distance_result_df["total_furlongs"] = (
    race_distance_result_df["miles"] * 8
    + race_distance_result_df["whole_furlongs"]
    + race_distance_result_df["has_half_furlong"].astype(int) * 0.5
)

race_distance_result_df["total_yards"] = (
    race_distance_result_df["total_furlongs"] * 220
).astype("Int64")

race_distance_result_df["parse_status"] = race_distance_result_df[
    "matches_revised_grammar"
].map({True: "parsed", False: "unresolved"})

race_level_reconciliation_df = pd.DataFrame(
    {
        "source_provisional_races": [len(provisional_race_distance_df)],
        "result_rows": [len(race_distance_result_df)],
        "unique_result_race_keys": [
            race_distance_result_df[["date", "course", "off"]]
            .drop_duplicates()
            .shape[0]
        ],
        "parsed_races": [
            int((race_distance_result_df["parse_status"] == "parsed").sum())
        ],
        "unresolved_races": [
            int((race_distance_result_df["parse_status"] == "unresolved").sum())
        ],
        "raw_distance_preserved_rows": [
            int(
                race_distance_result_df["raw_dist"].eq(
                    provisional_race_distance_df["raw_dist"]
                ).sum()
            )
        ],
        "missing_total_yards_rows": [
            int(race_distance_result_df["total_yards"].isna().sum())
        ],
    }
)

display(race_level_reconciliation_df)

display(
    race_distance_result_df.loc[
        race_distance_result_df["parse_status"] != "parsed"
    ].head(20)
)

,source_provisional_races,result_rows,unique_result_race_keys,parsed_races,unresolved_races,raw_distance_preserved_rows,missing_total_yards_rows
0,189043,189043,189043,189043,0,189043,0


,date,course,off,raw_dist,runner_records,matches_revised_grammar,miles,whole_furlongs,has_half_furlong,total_furlongs,total_yards,parse_status


## Full provisional-race reconciliation

### Observation

The validated parser was applied directly to all 189,043 provisional races.

The reconciliation is complete:

- 189,043 source provisional races;
- 189,043 parsed result rows;
- 189,043 unique provisional race keys;
- 189,043 races with `parse_status = parsed`;
- zero unresolved races;
- zero missing candidate yard values;
- exact raw `dist` preserved for every race.

No race was duplicated, dropped or left without a deterministic analytical distance.

### Interpretation

The candidate parsing rule now satisfies the principal coverage and preservation requirements for the current source extract.

The rule can:

1. preserve the exact source expression;
2. identify one observed imperial miles-and-furlongs notation family;
3. extract deterministic components;
4. represent every scheduled race distance exactly as integer yards;
5. expose unresolved future values through an explicit parse status rather than silently coercing them.

The current source contains no observed parse exceptions. The implementation should nevertheless retain an `unresolved` path because later or replacement source snapshots may introduce new notation.

### Provisional transformation rule

For each provisional race:

- preserve exact raw `dist`;
- accept only values matching the validated miles/furlongs/half-furlong grammar;
- parse whole miles, whole furlongs and the half-furlong indicator separately;
- calculate exact total yards;
- classify matching values as `parsed`;
- preserve non-matching future values unchanged and classify them as `unresolved`;
- never invent a distance for an unmatched value.

### Remaining evidence

Before closing the investigation, the study should confirm whether notation use varies materially by jurisdiction or race type. This is needed to distinguish a genuinely source-wide convention from a pattern driven mainly by UK and Irish racing.

In [10]:
# Test whether complete parser coverage holds across race types and the explicit
# jurisdiction suffixes carried in raw course names.
#
# This does not rebuild Notebook 04's full candidate jurisdiction mapping.
# It uses only terminal parenthetical course suffixes as direct source evidence
# and keeps courses without such a suffix in an explicit "<NO_SUFFIX>" group.

race_context_df = pd.read_sql_query(
    f"""
    SELECT
        date,
        course,
        off,
        MIN(type) AS race_type,
        MIN(dist) AS raw_dist
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY date, course, off
    """,
    connection,
)

# Extract the final parenthetical marker only, such as "(IRE)" or "(FR)".
# Meaningful configuration markers are retained as observed source values here;
# they are not automatically treated as jurisdictions.
race_context_df["terminal_course_suffix"] = (
    race_context_df["course"]
    .str.extract(r"(\([^()]+\))\s*$", expand=False)
    .fillna("<NO_SUFFIX>")
)

race_context_parsed_df = pd.DataFrame(
    classify_revised_raw_distance(raw_dist)
    for raw_dist in race_context_df["raw_dist"]
)

race_context_result_df = pd.concat(
    [
        race_context_df.reset_index(drop=True),
        race_context_parsed_df.drop(columns=["raw_dist"]).reset_index(drop=True),
    ],
    axis=1,
)

race_context_result_df["parse_status"] = race_context_result_df[
    "matches_revised_grammar"
].map({True: "parsed", False: "unresolved"})

race_type_coverage_df = (
    race_context_result_df.groupby("race_type", dropna=False)
    .agg(
        provisional_races=("raw_dist", "size"),
        distinct_raw_distances=("raw_dist", "nunique"),
        parsed_races=("matches_revised_grammar", "sum"),
    )
    .reset_index()
)

race_type_coverage_df["unresolved_races"] = (
    race_type_coverage_df["provisional_races"]
    - race_type_coverage_df["parsed_races"]
)

suffix_coverage_df = (
    race_context_result_df.groupby("terminal_course_suffix", dropna=False)
    .agg(
        provisional_races=("raw_dist", "size"),
        distinct_courses=("course", "nunique"),
        distinct_raw_distances=("raw_dist", "nunique"),
        parsed_races=("matches_revised_grammar", "sum"),
    )
    .reset_index()
)

suffix_coverage_df["unresolved_races"] = (
    suffix_coverage_df["provisional_races"]
    - suffix_coverage_df["parsed_races"]
)

display(
    race_type_coverage_df.sort_values(
        ["provisional_races", "race_type"],
        ascending=[False, True],
    )
)

display(
    suffix_coverage_df.sort_values(
        ["provisional_races", "terminal_course_suffix"],
        ascending=[False, True],
    )
)

,race_type,provisional_races,distinct_raw_distances,parsed_races,unresolved_races
1,Flat,126391,45,126391,0
2,Hurdle,35462,26,35462,0
0,Chase,22547,42,22547,0
3,NH Flat,4643,19,4643,0


,terminal_course_suffix,provisional_races,distinct_courses,distinct_raw_distances,parsed_races,unresolved_races
39,<NO_SUFFIX>,84895,189,61,84895,0
2,(AW),29705,7,19,29705,0
16,(IRE),29215,27,54,29215,0
11,(FR),19361,73,50,19361,0
14,(HK),6847,2,9,6847,0
38,(USA),5863,56,30,5863,0
1,(AUS),3798,51,20,3798,0
36,(UAE),2219,5,13,2219,0
19,(JPN),1470,21,22,1470,0
20,(July),1438,1,9,1438,0


## Coverage across race contexts

### Observation

The validated distance grammar parses every provisional race in all four observed race types:

- Flat: 126,391 of 126,391;
- Hurdle: 35,462 of 35,462;
- Chase: 22,547 of 22,547;
- NH Flat: 4,643 of 4,643.

It also parses every race across all observed terminal course-suffix groups, including international jurisdictions and courses without a terminal suffix.

No race-type or course-context group contains an unresolved distance value.

#### Interpretation

Across the current extract, the `dist` field uses one consistent imperial miles-and-furlongs notation family for all observed race types and course contexts.

This supports one common parser for the current source data.

It does not establish how the values were originally published, whether they were converted before entering this database, or whether any jurisdiction-specific rounding occurred.

### Decision

Use one source-wide distance parser for the current extract.

Do not add jurisdiction-specific parsing branches unless a later source snapshot introduces values that fail the common grammar.

This coverage question is resolved and requires no further jurisdiction profiling.

In [11]:
# Test exact round-trip reconstruction of the source notation.
#
# This verifies that the parsed components preserve enough information to
# reproduce every observed raw `dist` value exactly, including forms such as:
# - "2m"
# - "7½f"
# - "2m½f"
# - "2m3½f"
#
# Exact reconstruction is a useful protection against a parser that calculates
# the right total distance while losing how the source expressed it.

def reconstruct_raw_distance(
    miles: int,
    whole_furlongs: int,
    has_half_furlong: bool,
) -> str:
    """Reconstruct the validated source notation from parsed components."""
    miles_part = f"{miles}m" if miles > 0 else ""

    if whole_furlongs > 0:
        furlongs_part = f"{whole_furlongs}{'½' if has_half_furlong else ''}f"
    elif has_half_furlong:
        furlongs_part = "½f"
    else:
        furlongs_part = ""

    return f"{miles_part}{furlongs_part}"


round_trip_df = candidate_distance_df.copy()

round_trip_df["reconstructed_raw_dist"] = round_trip_df.apply(
    lambda row: reconstruct_raw_distance(
        miles=int(row["miles"]),
        whole_furlongs=int(row["whole_furlongs"]),
        has_half_furlong=bool(row["has_half_furlong"]),
    ),
    axis=1,
)

round_trip_df["exact_raw_round_trip"] = (
    round_trip_df["reconstructed_raw_dist"] == round_trip_df["raw_dist"]
)

round_trip_summary_df = pd.DataFrame(
    {
        "distinct_raw_values": [len(round_trip_df)],
        "exact_round_trip_values": [
            int(round_trip_df["exact_raw_round_trip"].sum())
        ],
        "failed_round_trip_values": [
            int((~round_trip_df["exact_raw_round_trip"]).sum())
        ],
        "exact_round_trip_provisional_races": [
            int(
                round_trip_df.loc[
                    round_trip_df["exact_raw_round_trip"],
                    "provisional_races",
                ].sum()
            )
        ],
        "failed_round_trip_provisional_races": [
            int(
                round_trip_df.loc[
                    ~round_trip_df["exact_raw_round_trip"],
                    "provisional_races",
                ].sum()
            )
        ],
    }
)

display(round_trip_summary_df)

display(
    round_trip_df.loc[
        ~round_trip_df["exact_raw_round_trip"],
        [
            "raw_dist",
            "miles",
            "whole_furlongs",
            "has_half_furlong",
            "reconstructed_raw_dist",
            "provisional_races",
        ],
    ].sort_values("raw_dist")
)

,distinct_raw_values,exact_round_trip_values,failed_round_trip_values,exact_round_trip_provisional_races,failed_round_trip_provisional_races
0,63,63,0,189043,0


,raw_dist,miles,whole_furlongs,has_half_furlong,reconstructed_raw_dist,provisional_races


## Round-trip preservation check

### Observation

All 63 distinct raw distance values can be reconstructed exactly from the parsed components, covering all 189,043 provisional races.

### Interpretation

The parser does not lose notation detail for the current source domain.

However, exact reconstruction is not required for source preservation because the original `dist` value will remain stored unchanged alongside any parsed fields.

### Decision

Treat this as a supporting validation only.

Do not extend the notebook with further notation-reconstruction tests unless a later source snapshot introduces values that fail the parser or lose information during parsing.

In [12]:
# Define the final candidate parser with an explicit unresolved path.
#
# The parser:
# - preserves the exact raw value;
# - extracts components only when the complete value matches the validated grammar;
# - derives exact integer yards for parsed values;
# - returns `unresolved` without inventing a distance for missing, malformed,
#   metric or otherwise unsupported future values.

def parse_race_distance(raw_dist) -> dict:
    """Parse one source race-distance value conservatively."""
    result = {
        "raw_dist": raw_dist,
        "notation_family": None,
        "source_unit": None,
        "miles": None,
        "whole_furlongs": None,
        "has_half_furlong": None,
        "total_furlongs": None,
        "total_yards": None,
        "parse_status": "unresolved",
    }

    if not isinstance(raw_dist, str) or raw_dist == "":
        return result

    match = REVISED_DISTANCE_PATTERN.fullmatch(raw_dist)

    if match is None:
        return result

    groups = match.groupdict()

    miles = int(groups["miles"]) if groups["miles"] is not None else 0

    if groups["furlongs_after_miles"] is not None:
        whole_furlongs = int(groups["furlongs_after_miles"])
    elif groups["furlongs_only"] is not None:
        whole_furlongs = int(groups["furlongs_only"])
    else:
        whole_furlongs = 0

    has_half_furlong = bool(
        groups["half_after_miles"]
        or groups["half_only_after_miles"]
        or groups["half_furlongs_only"]
    )

    # Reject syntactically matching but non-canonical component values.
    if whole_furlongs > 7:
        return result

    total_furlongs = (
        miles * 8
        + whole_furlongs
        + (0.5 if has_half_furlong else 0.0)
    )

    if total_furlongs <= 0:
        return result

    result.update(
        {
            "notation_family": "imperial_miles_furlongs",
            "source_unit": "miles_furlongs",
            "miles": miles,
            "whole_furlongs": whole_furlongs,
            "has_half_furlong": has_half_furlong,
            "total_furlongs": total_furlongs,
            "total_yards": int(total_furlongs * 220),
            "parse_status": "parsed",
        }
    )

    return result


# Test both observed valid forms and plausible unsupported future inputs.
parser_test_values = [
    "4f",
    "7½f",
    "1m",
    "2m½f",
    "2m3f",
    "3m4½f",
    "",
    None,
    "1600m",
    "1.5m",
    "1m8f",
    "unknown",
]

parser_test_df = pd.DataFrame(
    parse_race_distance(value)
    for value in parser_test_values
)

parser_test_df

,raw_dist,notation_family,source_unit,miles,whole_furlongs,has_half_furlong,total_furlongs,total_yards,parse_status
0,4f,imperial_miles_furlongs,miles_furlongs,0.0,4.0,False,4.0,880.0,parsed
1,7½f,imperial_miles_furlongs,miles_furlongs,0.0,7.0,True,7.5,1650.0,parsed
2,1m,imperial_miles_furlongs,miles_furlongs,1.0,0.0,False,8.0,1760.0,parsed
3,2m½f,imperial_miles_furlongs,miles_furlongs,2.0,0.0,True,16.5,3630.0,parsed
4,2m3f,imperial_miles_furlongs,miles_furlongs,2.0,3.0,False,19.0,4180.0,parsed
5,3m4½f,imperial_miles_furlongs,miles_furlongs,3.0,4.0,True,28.5,6270.0,parsed
6,,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
7,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
8,1600m,imperial_miles_furlongs,miles_furlongs,1600.0,0.0,False,12800.0,2816000.0,parsed
9,1.5m,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved


## Candidate parser failure and revised rule

### Observation

The candidate parser incorrectly accepted the test value `1600m` as 1,600 miles.

The syntax alone cannot distinguish the source notation `1m` meaning one mile from a possible metric expression such as `1600m` meaning 1,600 metres.

### Interpretation

A general regular-expression match is not sufficient evidence to assign a notation family or source unit to previously unseen values.

The current investigation has validated 63 exact raw values, not every possible string that happens to match the same character pattern.

### Revised transformation rule

For the current source extract:

- preserve every raw `dist` value unchanged;
- automatically parse the 63 exact values observed and validated in this notebook;
- derive exact total yards for those validated values;
- classify any previously unseen value as `unresolved`;
- do not infer whether an unseen `m` suffix means miles or metres;
- review and extend the validated domain deliberately when a later source snapshot introduces new values.

This is safer than applying an unrestricted grammar and prevents new notation from being silently misinterpreted.

In [13]:
# Build a validated lookup from the 63 exact source values observed in this
# notebook. Previously unseen values remain unresolved until deliberately
# reviewed and added to the validated domain.

validated_distance_lookup = {
    row["raw_dist"]: {
        "notation_family": "imperial_miles_furlongs",
        "source_unit": "miles_furlongs",
        "miles": int(row["miles"]),
        "whole_furlongs": int(row["whole_furlongs"]),
        "has_half_furlong": bool(row["has_half_furlong"]),
        "total_furlongs": float(row["total_furlongs"]),
        "total_yards": int(row["total_yards"]),
    }
    for _, row in candidate_distance_df.iterrows()
}


def parse_validated_race_distance(raw_dist) -> dict:
    """Parse only exact distance values validated from the current source."""
    result = {
        "raw_dist": raw_dist,
        "notation_family": None,
        "source_unit": None,
        "miles": None,
        "whole_furlongs": None,
        "has_half_furlong": None,
        "total_furlongs": None,
        "total_yards": None,
        "parse_status": "unresolved",
    }

    parsed = validated_distance_lookup.get(raw_dist)

    if parsed is None:
        return result

    result.update(parsed)
    result["parse_status"] = "parsed"
    return result


# Confirm complete coverage of the current source domain and conservative
# handling of plausible unseen values.
validated_parser_test_values = list(raw_distance_values_df["raw_dist"]) + [
    "",
    None,
    "1600m",
    "1.5m",
    "1m8f",
    "unknown",
]

validated_parser_test_df = pd.DataFrame(
    parse_validated_race_distance(value)
    for value in validated_parser_test_values
)

validated_parser_summary_df = pd.DataFrame(
    {
        "observed_distinct_values": [len(raw_distance_values_df)],
        "observed_values_parsed": [
            int(
                validated_parser_test_df.iloc[
                    : len(raw_distance_values_df)
                ]["parse_status"].eq("parsed").sum()
            )
        ],
        "observed_values_unresolved": [
            int(
                validated_parser_test_df.iloc[
                    : len(raw_distance_values_df)
                ]["parse_status"].eq("unresolved").sum()
            )
        ],
        "unseen_test_values": [
            len(validated_parser_test_values) - len(raw_distance_values_df)
        ],
        "unseen_test_values_unresolved": [
            int(
                validated_parser_test_df.iloc[
                    len(raw_distance_values_df) :
                ]["parse_status"].eq("unresolved").sum()
            )
        ],
    }
)

display(validated_parser_summary_df)
display(validated_parser_test_df.tail(6))

,observed_distinct_values,observed_values_parsed,observed_values_unresolved,unseen_test_values,unseen_test_values_unresolved
0,63,63,0,6,6


,raw_dist,notation_family,source_unit,miles,whole_furlongs,has_half_furlong,total_furlongs,total_yards,parse_status
63,,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
64,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
65,1600m,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
66,1.5m,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
67,1m8f,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved
68,unknown,NaN,NaN,NaN,NaN,None,NaN,NaN,unresolved


## Validated-domain parser result

### Observation

The conservative parser behaves as intended:

- all 63 observed raw distance values are parsed;
- zero observed values remain unresolved;
- all six deliberately unseen test values remain unresolved;
- the ambiguous test value `1600m` is no longer misclassified as miles.

### Interpretation

The validated-domain approach protects the current transformation from silently extending beyond the evidence.

The parser now has a clear failure condition:

> any raw distance value not present in the validated lookup must return `parse_status = unresolved`.

This means a future source snapshot can introduce new notation without being incorrectly converted.

### Final candidate transformation rule

For the current source extract:

1. preserve exact raw `dist`;
2. recognise the 63 validated raw values;
3. retain parsed miles, whole furlongs and half-furlong status separately;
4. calculate exact total furlongs and integer total yards;
5. assign `parse_status = parsed` only to validated values;
6. assign `parse_status = unresolved` to every unseen value;
7. never invent or infer a standard distance for an unresolved value.

### Database consequence

A later staging implementation should preserve the raw distance and add reversible parsed attributes separately.

The strongest candidate analytical standard is integer total yards because it exactly represents every validated current value without fractional storage.

### Next implementation step

The rule has reached the project stopping point: it is defensible, its known failure mode is explicit, unresolved future values are preserved, and validation can detect domain drift.

The stable parser and validated lookup can therefore be extracted into a focused reusable module with an independent validation script.

In [14]:
%%writefile ../src/inside_rails/race_distance.py
"""Validated parsing for scheduled race distances in the current source extract.

The source `dist` field is preserved unchanged. Only exact raw values that have
been reviewed and validated are converted. Previously unseen values remain
unresolved so that new notation cannot be silently misinterpreted.
"""

from __future__ import annotations

from typing import Any


# Exact values validated in Notebook 06.
#
# Each tuple contains:
#   miles,
#   whole furlongs,
#   half-furlong indicator
_VALIDATED_COMPONENTS: dict[str, tuple[int, int, bool]] = {
    raw_dist: (
        int(parsed["miles"]),
        int(parsed["whole_furlongs"]),
        bool(parsed["has_half_furlong"]),
    )
    for raw_dist, parsed in validated_distance_lookup.items()
}


def parse_race_distance(raw_dist: Any) -> dict[str, Any]:
    """Parse one exact source race-distance value conservatively.

    Values outside the validated source domain are preserved but returned with
    ``parse_status = "unresolved"`` and no inferred analytical distance.
    """
    result: dict[str, Any] = {
        "raw_dist": raw_dist,
        "notation_family": None,
        "source_unit": None,
        "miles": None,
        "whole_furlongs": None,
        "has_half_furlong": None,
        "total_furlongs": None,
        "total_yards": None,
        "parse_status": "unresolved",
    }

    components = _VALIDATED_COMPONENTS.get(raw_dist)

    if components is None:
        return result

    miles, whole_furlongs, has_half_furlong = components

    total_furlongs = (
        miles * 8
        + whole_furlongs
        + (0.5 if has_half_furlong else 0.0)
    )

    result.update(
        {
            "notation_family": "imperial_miles_furlongs",
            "source_unit": "miles_furlongs",
            "miles": miles,
            "whole_furlongs": whole_furlongs,
            "has_half_furlong": has_half_furlong,
            "total_furlongs": total_furlongs,
            "total_yards": int(total_furlongs * 220),
            "parse_status": "parsed",
        }
    )

    return result

Overwriting ../src/inside_rails/race_distance.py


In [15]:
# Rewrite the distance module so it is fully self-contained.
#
# This embeds the already validated 63-value lookup as literal Python data.
# The module will no longer depend on notebook variables when imported.

from pathlib import Path
from pprint import pformat

module_path = PROJECT_ROOT / "src" / "inside_rails" / "race_distance.py"

validated_components = {
    raw_dist: (
        int(parsed["miles"]),
        int(parsed["whole_furlongs"]),
        bool(parsed["has_half_furlong"]),
    )
    for raw_dist, parsed in validated_distance_lookup.items()
}

module_content = f'''"""Validated scheduled race-distance parsing.

The exact source value is preserved. Only raw distance values observed and
validated in Notebook 06 are converted. Unseen values remain unresolved.
"""

from __future__ import annotations

from typing import Any


VALIDATED_COMPONENTS: dict[str, tuple[int, int, bool]] = {
    pformat(validated_components, sort_dicts=True, width=88)
}


def parse_race_distance(raw_dist: Any) -> dict[str, Any]:
    """Parse one exact validated source race-distance value."""
    result: dict[str, Any] = {{
        "raw_dist": raw_dist,
        "miles": None,
        "whole_furlongs": None,
        "has_half_furlong": None,
        "total_furlongs": None,
        "total_yards": None,
        "distance_metres": None,
        "parse_status": "unresolved",
    }}

    components = VALIDATED_COMPONENTS.get(raw_dist)

    if components is None:
        return result

    miles, whole_furlongs, has_half_furlong = components

    total_furlongs = (
        miles * 8
        + whole_furlongs
        + (0.5 if has_half_furlong else 0.0)
    )

    total_yards = int(total_furlongs * 220)

    result.update(
        {{
            "miles": miles,
            "whole_furlongs": whole_furlongs,
            "has_half_furlong": has_half_furlong,
            "total_furlongs": total_furlongs,
            "total_yards": total_yards,
            "distance_metres": total_yards * 0.9144,
            "parse_status": "parsed",
        }}
    )

    return result
'''

module_path.write_text(module_content, encoding="utf-8")

print(f"Rewrote: {module_path}")
print(f"Embedded validated values: {len(validated_components)}")

Rewrote: /home/rob/Documents/inside-rails-horse-racing/src/inside_rails/race_distance.py
Embedded validated values: 63


In [16]:
# Verify that the standalone module now imports correctly and behaves as intended.

import importlib
import sys

src_path = str(PROJECT_ROOT / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

import inside_rails.race_distance as race_distance

race_distance = importlib.reload(race_distance)

print("Module imported successfully.")
print(f"Validated distance values: {len(race_distance.VALIDATED_COMPONENTS)}")
print()
print("Known value:")
print(race_distance.parse_race_distance("2m3½f"))
print()
print("Unseen value:")
print(race_distance.parse_race_distance("1600m"))


Module imported successfully.
Validated distance values: 63

Known value:
{'raw_dist': '2m3½f', 'miles': 2, 'whole_furlongs': 3, 'has_half_furlong': True, 'total_furlongs': 19.5, 'total_yards': 4290, 'distance_metres': 3922.776, 'parse_status': 'parsed'}

Unseen value:
{'raw_dist': '1600m', 'miles': None, 'whole_furlongs': None, 'has_half_furlong': None, 'total_furlongs': None, 'total_yards': None, 'distance_metres': None, 'parse_status': 'unresolved'}


In [17]:
# Validate the standalone module against every provisional race.
#
# This confirms complete current-source coverage and checks that the reusable
# module reproduces the notebook's established yard conversion exactly.

module_validation_df = provisional_race_distance_df.copy()

module_results_df = pd.DataFrame(
    race_distance.parse_race_distance(raw_dist)
    for raw_dist in module_validation_df["raw_dist"]
)

module_validation_df = pd.concat(
    [
        module_validation_df.reset_index(drop=True),
        module_results_df.drop(columns=["raw_dist"]).reset_index(drop=True),
    ],
    axis=1,
)

module_validation_df["expected_total_yards"] = (
    module_validation_df["raw_dist"]
    .map(validated_distance_lookup)
    .map(lambda parsed: int(parsed["total_yards"]))
)

module_validation_df["yards_reconcile"] = (
    module_validation_df["total_yards"]
    == module_validation_df["expected_total_yards"]
)

module_validation_summary_df = pd.DataFrame(
    {
        "provisional_races": [len(module_validation_df)],
        "parsed_races": [
            int(module_validation_df["parse_status"].eq("parsed").sum())
        ],
        "unresolved_races": [
            int(module_validation_df["parse_status"].eq("unresolved").sum())
        ],
        "yards_reconciling_races": [
            int(module_validation_df["yards_reconcile"].sum())
        ],
        "yards_non_reconciling_races": [
            int((~module_validation_df["yards_reconcile"]).sum())
        ],
        "missing_metres_rows": [
            int(module_validation_df["distance_metres"].isna().sum())
        ],
    }
)

display(module_validation_summary_df)
display(
    module_validation_df.loc[
        ~module_validation_df["yards_reconcile"]
        | module_validation_df["parse_status"].ne("parsed")
    ].head(20)
)

,provisional_races,parsed_races,unresolved_races,yards_reconciling_races,yards_non_reconciling_races,missing_metres_rows
0,189043,189043,0,189043,0,0


,date,course,off,raw_dist,runner_records,miles,whole_furlongs,has_half_furlong,total_furlongs,total_yards,distance_metres,parse_status,expected_total_yards,yards_reconcile


## Reusable-module validation

### Observation

The standalone distance module was applied to all 189,043 provisional races.

The validation passed completely:

- 189,043 races parsed;
- zero unresolved current-source races;
- 189,043 exact yard reconciliations;
- zero yard mismatches;
- zero missing metre conversions.

### Interpretation

The reusable implementation reproduces the notebook’s established conversion rule across the complete current source population.

The current distance field can therefore be represented reproducibly by preserving the exact raw `dist` value and adding:

- parsed miles;
- parsed whole furlongs;
- half-furlong indicator;
- total furlongs;
- exact integer total yards;
- derived metres;
- parse status.

### Decision

Use integer yards as the exact standard analytical distance.

Use metres as the SI-derived value calculated from yards.

Preserve the raw `dist` field unchanged.

Any future raw value outside the validated 63-value domain must remain unresolved until reviewed rather than being interpreted automatically.

In [18]:
# Create an independent validation script for the reusable distance parser.
#
# The script opens the immutable source database in read-only mode, reconstructs
# provisional races, checks complete parser coverage, and verifies exact yards
# and metre availability across the current source population.

validation_script_path = (
    PROJECT_ROOT / "scripts" / "validate_race_distance.py"
)

validation_script_content = '''"""Validate scheduled race-distance parsing against the source database."""

from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from inside_rails.race_distance import parse_race_distance


DATA_ROW_PREDICATE = "rowid <> 1"


def main() -> None:
    database_path = (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "form_2015-present"
        / "form_2015-present"
        / "raceform.db"
    )

    if not database_path.exists():
        raise FileNotFoundError(f"Source database not found: {database_path}")

    connection = sqlite3.connect(
        f"file:{database_path}?mode=ro",
        uri=True,
    )

    try:
        provisional_races = pd.read_sql_query(
            f"""
            SELECT
                date,
                course,
                off,
                MIN(dist) AS raw_dist,
                COUNT(DISTINCT dist) AS distinct_distance_values
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY date, course, off
            """,
            connection,
        )
    finally:
        connection.close()

    results = pd.DataFrame(
        parse_race_distance(raw_dist)
        for raw_dist in provisional_races["raw_dist"]
    )

    assert len(provisional_races) == 189_043
    assert provisional_races["distinct_distance_values"].eq(1).all()
    assert results["parse_status"].eq("parsed").all()
    assert results["total_yards"].notna().all()
    assert results["distance_metres"].notna().all()
    assert results["total_yards"].between(880, 8_030).all()

    # Confirm that unseen values are not interpreted automatically.
    assert parse_race_distance("1600m")["parse_status"] == "unresolved"
    assert parse_race_distance(None)["parse_status"] == "unresolved"

    print("Race-distance validation passed.")
    print(f"Provisional races checked: {len(provisional_races):,}")
    print(f"Distinct raw values: {provisional_races['raw_dist'].nunique():,}")
    print(f"Parsed races: {results['parse_status'].eq('parsed').sum():,}")
    print(f"Unresolved races: {results['parse_status'].eq('unresolved').sum():,}")


if __name__ == "__main__":
    main()
'''

validation_script_path.write_text(
    validation_script_content,
    encoding="utf-8",
)

print(f"Created: {validation_script_path}")

Created: /home/rob/Documents/inside-rails-horse-racing/scripts/validate_race_distance.py


In [19]:
# Run the independent validation script from the project environment.

import subprocess

validation_result = subprocess.run(
    [
        str(PROJECT_ROOT / ".venv" / "bin" / "python"),
        str(PROJECT_ROOT / "scripts" / "validate_race_distance.py"),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(validation_result.stdout)

if validation_result.stderr:
    print(validation_result.stderr)

assert validation_result.returncode == 0, (
    f"Validation script failed with exit code "
    f"{validation_result.returncode}"
)

Race-distance validation passed.
Provisional races checked: 189,043
Distinct raw values: 63
Parsed races: 189,043
Unresolved races: 0



# Conclusion

## Answer

The source `dist` field can be standardised reproducibly for analysis while preserving exactly how each race distance was expressed.

## Evidence

- All 1,851,285 runner records contain a text distance value.
- All 189,043 provisional races contain one internally consistent raw `dist` value.
- The source contains 63 distinct raw distance values.
- Every observed value uses miles-and-furlongs notation, including optional half-furlongs represented by `½`.
- All 63 values parse deterministically.
- All 189,043 provisional races convert exactly to integer yards.
- Derived metres are available for every race.
- The independent validation script passed with zero unresolved current-source races.

## Interpretation

The exact raw distance should remain the authoritative source expression.

Integer yards provide the strongest exact analytical representation because every observed half-furlong corresponds to 110 yards and no fractional yards are required.

Metres are useful as the SI analytical distance, but they are derived from the source’s imperial representation rather than supplied directly by the source.

The evidence does not establish how distances were originally published in each jurisdiction or whether any upstream conversion or rounding occurred before the values entered this database.

## Database consequence

A later staging implementation should preserve:

- raw `dist`;
- parsed miles;
- parsed whole furlongs;
- half-furlong indicator;
- total furlongs;
- exact integer total yards;
- derived distance in metres;
- parse status.

The current 63-value domain may be parsed automatically.

Any previously unseen raw value must remain unresolved until reviewed. The system must not infer whether an unfamiliar suffix such as `m` means miles or metres.

## Confidence

**High** for parsing and conversion of the current source extract.

**Unknown** for the provenance, original publication units and possible upstream rounding of international race distances.

## Stopping decision

The bounded question is resolved.

The transformation rule is explicit, complete for the current source population, reversible, preserves unresolved future values and is protected by an independent validation script. No further distance profiling is required before proceeding.

## Additional validation required: international distance conversion

### Issue identified

The source expresses every observed `dist` value in miles-and-furlongs notation, including races from jurisdictions that commonly publish scheduled distances in metres.

Therefore, converting the source value to metres does not necessarily recover the original official metric distance exactly.

For example, a race officially scheduled over 1,600 metres could potentially appear as `1m` in the source. Direct conversion of one mile produces 1,609.344 metres, which would overstate the official scheduled distance by 9.344 metres.

### Revised question

Before accepting derived metres as an analytical race distance, determine:

> Do the source’s miles-and-furlongs values preserve international scheduled distances exactly, or are metric races rounded into approximate imperial categories?

### Method

The study will not manually verify every race individually.

Instead it will:

1. isolate jurisdictions that commonly use metric scheduled distances;
2. profile the imperial values used for those races;
3. calculate the implied metre values;
4. look for repeated mappings consistent with standard metric distances such as 1,000m, 1,200m, 1,400m, 1,600m, 2,000m and 2,400m;
5. externally verify representative races and any ambiguous mappings;
6. classify derived metres as exact, approximate or unresolved according to the evidence.

### Consequence

Until this validation is complete:

- raw `dist` remains valid source evidence;
- exact integer yards represent the source expression reproducibly;
- derived metres must not yet be described as the original official scheduled distance;
- the distance investigation is not closed.

In [20]:
# Isolate races from jurisdictions that normally publish scheduled distances
# metrically, then profile how their official-looking metric distances may have
# been represented by the source's miles-and-furlongs `dist` field.
#
# This is an investigative grouping only. It does not yet claim that every
# race in each jurisdiction was officially advertised in metres.

METRIC_JURISDICTION_SUFFIXES = {
    "(ARG)", "(AUS)", "(BEL)", "(BHR)", "(BRZ)", "(CAN)", "(CHI)",
    "(CHN)", "(CZE)", "(DEN)", "(FR)", "(GER)", "(HK)", "(HUN)",
    "(ITY)", "(JPN)", "(KOR)", "(KSA)", "(NOR)", "(NZ)", "(PER)",
    "(POL)", "(QA)", "(SAF)", "(SIN)", "(SPA)", "(SWE)", "(SWI)",
    "(TUR)", "(UAE)", "(URU)",
}

metric_context_df = race_context_result_df.loc[
    race_context_result_df["terminal_course_suffix"].isin(
        METRIC_JURISDICTION_SUFFIXES
    )
].copy()

# Attach the already validated source-distance conversion.
metric_context_df["source_total_yards"] = (
    metric_context_df["raw_dist"]
    .map(validated_distance_lookup)
    .map(lambda parsed: int(parsed["total_yards"]))
)

metric_context_df["source_implied_metres"] = (
    metric_context_df["source_total_yards"] * 0.9144
)

metric_jurisdiction_summary_df = (
    metric_context_df.groupby("terminal_course_suffix")
    .agg(
        provisional_races=("raw_dist", "size"),
        distinct_courses=("course", "nunique"),
        distinct_source_distances=("raw_dist", "nunique"),
        minimum_implied_metres=("source_implied_metres", "min"),
        maximum_implied_metres=("source_implied_metres", "max"),
    )
    .reset_index()
    .sort_values(
        ["provisional_races", "terminal_course_suffix"],
        ascending=[False, True],
    )
)

metric_distance_frequency_df = (
    metric_context_df.groupby(
        ["raw_dist", "source_total_yards", "source_implied_metres"]
    )
    .size()
    .reset_index(name="provisional_races")
    .sort_values(
        ["provisional_races", "source_total_yards"],
        ascending=[False, True],
    )
)

print(
    f"Candidate metric-jurisdiction races: "
    f"{len(metric_context_df):,}"
)
print(
    f"Distinct source distance values represented: "
    f"{metric_context_df['raw_dist'].nunique():,}"
)

display(metric_jurisdiction_summary_df)
display(metric_distance_frequency_df)

Candidate metric-jurisdiction races: 37,472
Distinct source distance values represented: 51


,terminal_course_suffix,provisional_races,distinct_courses,distinct_source_distances,minimum_implied_metres,maximum_implied_metres
10,(FR),19361,73,50,804.672,7342.632
12,(HK),6847,2,9,1005.840,2414.016
1,(AUS),3798,51,20,905.256,3218.688
29,(UAE),2219,5,13,1005.840,3218.688
15,(JPN),1470,21,22,1005.840,4425.696
11,(GER),710,17,20,1005.840,4023.360
5,(CAN),462,4,14,905.256,2816.352
23,(SAF),453,9,14,1005.840,3218.688
0,(ARG),416,3,10,1005.840,2514.600
14,(ITY),372,7,16,1005.840,5029.200


,raw_dist,source_total_yards,source_implied_metres,provisional_races
0,1m,1760,1609.344,6658
47,6f,1320,1207.008,5242
49,7f,1540,1408.176,3796
3,1m2f,2200,2011.680,3022
2,1m1½f,2090,1911.096,1981
1,1m1f,1980,1810.512,1866
7,1m4f,2640,2414.016,1788
45,5f,1100,1005.840,1458
19,2m2f,3960,3621.024,1303
48,6½f,1430,1307.592,1280


In [21]:
# Select a small, reproducible verification sample from major metric jurisdictions
# and common source distance values.
#
# One race is selected for each jurisdiction-and-distance combination so that
# the race can be checked against an authoritative external result source.
# This is targeted validation, not an attempt to research all 37,472 races.

VERIFICATION_JURISDICTIONS = {
    "(FR)",
    "(HK)",
    "(AUS)",
    "(UAE)",
    "(JPN)",
    "(GER)",
}

VERIFICATION_RAW_DISTANCES = {
    "5f",
    "6f",
    "7f",
    "1m",
    "1m2f",
    "1m4f",
}

verification_candidates_df = pd.read_sql_query(
    f"""
    WITH provisional_races AS (
        SELECT
            date,
            course,
            off,
            MIN(race_name) AS race_name,
            MIN(type) AS race_type,
            MIN(dist) AS raw_dist,
            MIN(race_id) AS source_race_id,
            COUNT(*) AS runner_records
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY date, course, off
    )
    SELECT *
    FROM provisional_races
    """,
    connection,
)

verification_candidates_df["terminal_course_suffix"] = (
    verification_candidates_df["course"]
    .str.extract(r"(\([^()]+\))\s*$", expand=False)
    .fillna("<NO_SUFFIX>")
)

verification_sample_df = (
    verification_candidates_df.loc[
        verification_candidates_df["terminal_course_suffix"].isin(
            VERIFICATION_JURISDICTIONS
        )
        & verification_candidates_df["raw_dist"].isin(
            VERIFICATION_RAW_DISTANCES
        )
    ]
    .sort_values(
        [
            "terminal_course_suffix",
            "raw_dist",
            "date",
            "course",
            "off",
        ]
    )
    .groupby(
        ["terminal_course_suffix", "raw_dist"],
        as_index=False,
        group_keys=False,
    )
    .head(1)
    .reset_index(drop=True)
)

verification_sample_df[
    [
        "terminal_course_suffix",
        "date",
        "course",
        "off",
        "race_name",
        "race_type",
        "raw_dist",
        "source_race_id",
        "runner_records",
    ]
]

,terminal_course_suffix,date,course,off,race_name,race_type,raw_dist,source_race_id,runner_records
0,(AUS),2015-01-09,Gosford (AUS),5:45,Belle Of The Turf Stakes (Fillies & Mares) (H...,Flat,1m,617473,8
1,(AUS),2015-03-07,Randwick (AUS),3:10,Girvan Waugh Partners Randwick City Stakes (Li...,Flat,1m2f,620758,11
2,(AUS),2015-01-01,Ascot (AUS),8:50,Golden River Development Perth Cup (Handicap),Flat,1m4f,617103,16
3,(AUS),2015-01-26,Sandown (AUS),4:05,Sportingbet Powered by William Hill Blue Diamo...,Flat,5f,618276,15
4,(AUS),2015-01-01,Flemington (AUS),5:20,Standish Handicap,Flat,6f,617101,11
5,(AUS),2015-02-14,Caulfield (AUS),6:15,Hyland Race Colours Autumn Stakes (Turf),Flat,7f,619433,14
6,(FR),2015-01-14,Cagnes-Sur-Mer (FR),1:20,Prix de Caucade (Maiden) (Unraced 3yo Fillies)...,Flat,1m,617515,10
7,(FR),2015-01-13,Cagnes-Sur-Mer (FR),12:30,Prix de Carnoles (Maiden) (3yo Fillies) (Polyt...,Flat,1m2f,617494,15
8,(FR),2015-03-07,Saint-Cloud (FR),3:55,Prix dHerblay (Conditions) (4yo+) (Amateur Rid...,Flat,1m4f,620958,11
9,(FR),2015-04-02,Maisons-Laffitte (FR),11:45,Prix dOrgeval (Claimer) (2yo) (Turf),Flat,5f,623188,10


## External verification: metric races were approximated

### Observation

Authoritative race records confirm that at least some international metric races were represented approximately in the source `dist` field.

Verified examples:

| Date | Course | Race | Source `dist` | Official distance | Literal source conversion |
|---|---|---|---:|---:|---:|
| 2015-01-25 | Sha Tin (HK) | Stewards' Cup | `1m` | 1,600m | 1,609.344m |
| 2015-01-04 | Kyoto (JPN) | Sports Nippon Sho Kyoto Kimpai | `1m` | 1,600m | 1,609.344m |

Official sources:

- Hong Kong Jockey Club, *The Stewards' Cup — 1600 Metres — Sha Tin — Sunday 25 January 2015*
- Japan Racing Association, *2015 Kyoto Kimpai — course 1,600 metres*

### Interpretation

For these verified races, the source `dist` value is not an exact preservation of the official scheduled distance.

Instead, the official 1,600-metre distance has been represented as the nearby imperial category `1m`.

Consequently:

- `1m` remains an exact representation of what this source recorded;
- `1,760` yards is an exact conversion of that source expression;
- `1,609.344` metres is only the literal SI conversion of the source expression;
- it must not be presented as the verified official scheduled distance.

The evidence supports upstream standardisation or approximation, but does not yet identify whether it was performed by the original racing-data provider, an intermediate website or the dataset creator.

### Revised database consequence

Preserve a clear distinction between:

1. **raw source distance** — exact `dist` text;
2. **source-implied yards/metres** — deterministic conversion of that text;
3. **official scheduled distance** — separate enrichment requiring authoritative evidence.

Do not label the source-implied metre value simply as `distance_metres`, because that would imply an exact official measurement. A name such as `source_implied_metres` is more accurate until official distance enrichment is available.

## Scope decision for initial database use

### Decision

The initial analytical database will prioritise UK and Irish racing.

For those races, the source `dist` field can be retained and converted into source-implied yards and metres for early analytical work, subject to the normal preservation of the raw value.

International races will not be discarded. They will remain available with:

- exact raw `dist`;
- source-implied yards;
- source-implied metres;
- an explicit indication that the official scheduled distance has not yet been independently verified.

### Deferred enrichment

Exact official distances for metric jurisdictions will be treated as a separate enrichment workstream.

That enrichment may later use:

- other supplied source products;
- official racing-authority results;
- a suitably licensed international results source;
- race-level matching by date, course, off time and race name.

### Practical consequence

Unresolved international distance provenance does not need to delay construction or early use of the database.

Analyses requiring exact official distance should initially be limited to jurisdictions where the source representation is sufficiently trusted, or should explicitly use the source-implied distance rather than claim an official measurement.

In [22]:
# Rename the derived metric field so it does not imply an independently verified
# official scheduled distance.
#
# The module will now return:
# - source_implied_yards
# - source_implied_metres
#
# These are exact conversions of the raw source expression, not necessarily the
# original official distance for international metric races.

module_path = PROJECT_ROOT / "src" / "inside_rails" / "race_distance.py"
module_text = module_path.read_text(encoding="utf-8")

module_text = module_text.replace(
    '"total_yards": None,\n        "distance_metres": None,',
    '"source_implied_yards": None,\n        "source_implied_metres": None,',
)

module_text = module_text.replace(
    '"total_yards": total_yards,\n            "distance_metres": total_yards * 0.9144,',
    '"source_implied_yards": total_yards,\n'
    '            "source_implied_metres": total_yards * 0.9144,',
)

module_path.write_text(module_text, encoding="utf-8")

# Reload and inspect one known value.
race_distance = importlib.reload(race_distance)

print(f"Updated: {module_path}")
print(race_distance.parse_race_distance("1m"))

Updated: /home/rob/Documents/inside-rails-horse-racing/src/inside_rails/race_distance.py
{'raw_dist': '1m', 'miles': 1, 'whole_furlongs': 0, 'has_half_furlong': False, 'total_furlongs': 8.0, 'source_implied_yards': 1760, 'source_implied_metres': 1609.344, 'parse_status': 'parsed'}


In [23]:
# Rewrite the reusable distance module with explicit warnings that derived
# yards and metres describe the source expression, not necessarily the
# independently verified official scheduled distance.

from pathlib import Path
from pprint import pformat

module_path = PROJECT_ROOT / "src" / "inside_rails" / "race_distance.py"

validated_components = {
    raw_dist: (
        int(parsed["miles"]),
        int(parsed["whole_furlongs"]),
        bool(parsed["has_half_furlong"]),
    )
    for raw_dist, parsed in validated_distance_lookup.items()
}

module_content = f'''"""Validated scheduled race-distance parsing.

The exact raw source value is preserved.

Derived yards and metres are literal conversions of the source expression
only. They are not guaranteed to equal the original official scheduled
distance for jurisdictions that publish races metrically, because an upstream
provider may have rounded or standardised those distances into
miles-and-furlongs notation.

Only exact raw values observed and validated in Notebook 06 are converted.
Previously unseen values remain unresolved.
"""

from __future__ import annotations

from typing import Any


# Exact raw values validated in Notebook 06.
# Tuple structure: miles, whole furlongs, half-furlong indicator.
VALIDATED_COMPONENTS: dict[str, tuple[int, int, bool]] = {
    pformat(validated_components, sort_dicts=True, width=88)
}


def parse_race_distance(raw_dist: Any) -> dict[str, Any]:
    """Parse one exact validated source race-distance value.

    ``source_implied_yards`` and ``source_implied_metres`` describe the literal
    source expression. They must not be treated as independently verified
    official race distances.
    """
    result: dict[str, Any] = {{
        "raw_dist": raw_dist,
        "miles": None,
        "whole_furlongs": None,
        "has_half_furlong": None,
        "total_furlongs": None,
        "source_implied_yards": None,
        "source_implied_metres": None,
        "official_distance_verified": False,
        "parse_status": "unresolved",
    }}

    components = VALIDATED_COMPONENTS.get(raw_dist)

    if components is None:
        return result

    miles, whole_furlongs, has_half_furlong = components

    total_furlongs = (
        miles * 8
        + whole_furlongs
        + (0.5 if has_half_furlong else 0.0)
    )

    source_implied_yards = int(total_furlongs * 220)

    result.update(
        {{
            "miles": miles,
            "whole_furlongs": whole_furlongs,
            "has_half_furlong": has_half_furlong,
            "total_furlongs": total_furlongs,
            "source_implied_yards": source_implied_yards,
            "source_implied_metres": source_implied_yards * 0.9144,
            "official_distance_verified": False,
            "parse_status": "parsed",
        }}
    )

    return result
'''

module_path.write_text(module_content, encoding="utf-8")

race_distance = importlib.reload(race_distance)

print(f"Rewrote: {module_path}")
print(race_distance.parse_race_distance("1m"))

Rewrote: /home/rob/Documents/inside-rails-horse-racing/src/inside_rails/race_distance.py
{'raw_dist': '1m', 'miles': 1, 'whole_furlongs': 0, 'has_half_furlong': False, 'total_furlongs': 8.0, 'source_implied_yards': 1760, 'source_implied_metres': 1609.344, 'official_distance_verified': False, 'parse_status': 'parsed'}


In [24]:
# Update the independent validation script to match the clarified field names
# and confirm that source-implied distances are never presented as verified
# official distances.

validation_script_path = (
    PROJECT_ROOT / "scripts" / "validate_race_distance.py"
)

validation_script_content = '''"""Validate scheduled race-distance parsing against the source database."""

from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from inside_rails.race_distance import parse_race_distance


DATA_ROW_PREDICATE = "rowid <> 1"


def main() -> None:
    database_path = (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "form_2015-present"
        / "form_2015-present"
        / "raceform.db"
    )

    if not database_path.exists():
        raise FileNotFoundError(f"Source database not found: {database_path}")

    connection = sqlite3.connect(
        f"file:{database_path}?mode=ro",
        uri=True,
    )

    try:
        provisional_races = pd.read_sql_query(
            f"""
            SELECT
                date,
                course,
                off,
                MIN(dist) AS raw_dist,
                COUNT(DISTINCT dist) AS distinct_distance_values
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY date, course, off
            """,
            connection,
        )
    finally:
        connection.close()

    results = pd.DataFrame(
        parse_race_distance(raw_dist)
        for raw_dist in provisional_races["raw_dist"]
    )

    assert len(provisional_races) == 189_043
    assert provisional_races["distinct_distance_values"].eq(1).all()
    assert provisional_races["raw_dist"].nunique() == 63

    assert results["parse_status"].eq("parsed").all()
    assert results["source_implied_yards"].notna().all()
    assert results["source_implied_metres"].notna().all()

    assert results["source_implied_yards"].between(880, 8_030).all()

    # Source-implied conversions are not independently verified official values.
    assert results["official_distance_verified"].eq(False).all()

    # Confirm that unseen or unsupported values are not interpreted automatically.
    assert parse_race_distance("1600m")["parse_status"] == "unresolved"
    assert parse_race_distance(None)["parse_status"] == "unresolved"

    print("Race-distance validation passed.")
    print(f"Provisional races checked: {len(provisional_races):,}")
    print(f"Distinct raw values: {provisional_races['raw_dist'].nunique():,}")
    print(f"Parsed races: {results['parse_status'].eq('parsed').sum():,}")
    print(f"Unresolved races: {results['parse_status'].eq('unresolved').sum():,}")
    print(
        "Official-distance verified rows: "
        f"{results['official_distance_verified'].sum():,}"
    )


if __name__ == "__main__":
    main()
'''

validation_script_path.write_text(
    validation_script_content,
    encoding="utf-8",
)

print(f"Updated: {validation_script_path}")

Updated: /home/rob/Documents/inside-rails-horse-racing/scripts/validate_race_distance.py


In [25]:
# Run the updated independent validation script.

validation_result = subprocess.run(
    [
        str(PROJECT_ROOT / ".venv" / "bin" / "python"),
        str(PROJECT_ROOT / "scripts" / "validate_race_distance.py"),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(validation_result.stdout)

if validation_result.stderr:
    print(validation_result.stderr)

assert validation_result.returncode == 0, (
    f"Validation script failed with exit code "
    f"{validation_result.returncode}"
)

Race-distance validation passed.
Provisional races checked: 189,043
Distinct raw values: 63
Parsed races: 189,043
Unresolved races: 0
Official-distance verified rows: 0



# Conclusion

## Answer

The source `dist` field is complete, internally consistent and reproducibly parseable for the current extract.

All 189,043 provisional races use one of 63 validated miles-and-furlongs expressions. These values can be converted exactly into source-implied yards and deterministically into source-implied metres.

However, those conversions represent what the source expressed. They do not necessarily reproduce the original official scheduled distance for international races from jurisdictions that commonly publish distances metrically.

## Evidence

- 1,851,285 runner records contain a non-blank text `dist` value.
- Every provisional race contains one consistent raw distance.
- There are 63 distinct raw distance values.
- All 63 values parse successfully.
- All 189,043 provisional races receive source-implied yards and metres.
- Zero current-source races remain unresolved.
- The standalone validation script passes across the complete race population.
- All returned records explicitly have `official_distance_verified = False`.
- External checks found metric races officially scheduled over 1,600 metres represented by the source as `1m`.

## Interpretation

The raw `dist` value is reliable evidence of the source representation.

`source_implied_yards` is an exact conversion of that representation.

`source_implied_metres` is a literal SI conversion of the source representation, but it must not be described as the independently verified official distance.

For UK and Irish racing, the source representation is suitable for initial analytical work, while retaining the raw value and the same provenance warning.

International official distances require a separate enrichment process where exact jurisdiction-native distances are analytically important.

## Database consequence

A later staging implementation should preserve:

- exact raw `dist`;
- parsed miles;
- parsed whole furlongs;
- half-furlong indicator;
- total furlongs;
- source-implied yards;
- source-implied metres;
- parse status;
- official-distance verification status;
- separately enriched official distance and unit when available.

Previously unseen distance values must remain unresolved until reviewed.

An official distance must never be populated merely by converting the source expression.

## Confidence

**High** for parsing and literal conversion of the current source representation.

**Low or unknown** for treating source-implied values as exact official distances outside jurisdictions where the source convention has been independently validated.

## Stopping decision

The bounded parsing question is resolved.

The current source domain is completely parseable, the limitation is explicitly preserved, future domain drift is detectable, and international official-distance enrichment can proceed as a separate later workstream without delaying initial database construction.